# Instrument Server Manual Example

This notebook is a cell-by-cell manual example for calling the `instrument_server.py` Flask server from a client machine.

The instrument server wraps selected methods from `Automation` and exposes them through:

- `/ping`
- `/call_auto`
- `/last_export`
- `/download_last_export`
- `/download_file`

Only methods explicitly allowed in `ALLOWED_AUTO_METHODS` on the server can be called remotely.

## Cell 1: Imports and client class

Run this first. Change the token only if you changed it in `instrument_server.py`.

In [ ]:
import requests
from pathlib import Path


class InstrumentClient:
    def __init__(self, host, port=5000, token="SuperSecret_Token_41126est"):
        self.base_url = f"http://{host}:{port}"
        self.headers = {"X-API-Token": token}

    def ping(self):
        r = requests.get(
            f"{self.base_url}/ping",
            headers=self.headers,
            timeout=10,
        )
        r.raise_for_status()
        return r.json()

    def call_auto(self, method, *args, **kwargs):
        payload = {
            "method": method,
            "args": list(args),
            "kwargs": kwargs,
        }

        r = requests.post(
            f"{self.base_url}/call_auto",
            json=payload,
            headers=self.headers,
            timeout=600,
        )

        try:
            out = r.json()
        except Exception:
            r.raise_for_status()
            raise

        if r.status_code != 200 or out.get("status") != "ok":
            raise RuntimeError(out)

        return out["result"]

    def last_export(self):
        r = requests.get(
            f"{self.base_url}/last_export",
            headers=self.headers,
            timeout=20,
        )
        r.raise_for_status()
        return r.json()

    def download_last_export(self, local_dir="."):
        local_dir = Path(local_dir)
        local_dir.mkdir(parents=True, exist_ok=True)

        info = self.last_export()
        local_path = local_dir / info["name"]

        r = requests.get(
            f"{self.base_url}/download_last_export",
            headers=self.headers,
            timeout=120,
        )
        r.raise_for_status()

        local_path.write_bytes(r.content)
        return local_path

    def download_file(self, directory, filename, local_dir="."):
        local_dir = Path(local_dir)
        local_dir.mkdir(parents=True, exist_ok=True)

        r = requests.post(
            f"{self.base_url}/download_file",
            json={"directory": directory, "filename": filename},
            headers=self.headers,
            timeout=120,
        )
        r.raise_for_status()

        local_path = local_dir / filename
        local_path.write_bytes(r.content)
        return local_path

## Cell 2: Connect to instrument server

Replace `HOST` with the IP address of the instrument PC running `instrument_server.py`.

In [ ]:
HOST = "192.168.1.100"   # change this to the instrument PC IP
PORT = 5000

client = InstrumentClient(host=HOST, port=PORT)
client.ping()

## Cell 3: Read current UI state

In [ ]:
state = client.call_auto("get_all_ui_state")
state

In [ ]:
xyz = client.call_auto("get_xyz_positions")
xyz

Expected `xyz` return:

```python
# (x_position_um, y_position_um, extension_mm)
```

## Cell 4: Capture current sample/test info using OCR

In [ ]:
info = client.call_auto(
    "get_current_sample_and_test_ocr",
    debug=False,
)

info

## Cell 5: Capture video panel remotely

In [ ]:
result = client.call_auto(
    "capture_video_panel_remote",
    save_path=r"C:\Users\Public\Documents\Nanomechanics\Profiles\Shared AENI\remote_video_panel.png",
)

result

In [ ]:
local_img = client.download_file(
    directory=r"C:\Users\Public\Documents\Nanomechanics\Profiles\Shared AENI",
    filename="remote_video_panel.png",
    local_dir="downloads",
)

local_img

## Cell 6: Change method file

In [ ]:
client.call_auto(
    "change_method",
    method_dir=r"C:\Program Files\Nanomechanics\MasterMethods v1.5",
    method_file_name="Advanced Dynamic E and H.NMT",
    add=1,
)

## Cell 7: Move XY stage

In [ ]:
# Single move
client.call_auto(
    "move",
    amount=50,
    direction="right",
    t=1,
    tt=4,
    Backlash=None,
)

In [ ]:
# Move in increments
client.call_auto(
    "move_in_increments",
    total_amount=500,
    direction="left",
    increment=100,
    t=1,
    tt=4,
    Backlash=None,
)

In [ ]:
# Allowed directions
allowed_directions = ["left", "right", "up", "down"]
allowed_directions

## Cell 8: Set extension / engage / focus

In [ ]:
client.call_auto(
    "set_extension",
    number=10.0,
    t=2,
)

In [ ]:
client.call_auto("engage")

In [ ]:
# This is manual because it asks you to move to three positions on the instrument PC.
plane = client.call_auto("align_focus")
plane

In [ ]:
client.call_auto(
    "focus",
    plane_params=plane,
)

## Cell 9: Define EH setup and indent array

In [ ]:
setup = {
    "poisson": 0.34,
    "drift": 1,
    "metal": 1,
    "target load": 50,
    "target depth": 2000,
    "surface approach distance": 1000,
    "surface approach velocity": 100,
    "strain rate": 0.05,
}

indents = {
    "X Indents": 3,
    "Y Indents": 3,
    "X Spacing": 50,
    "Y Spacing": 50,
    "Rotation": 0,
}

setup, indents

## Cell 10: Start a test setup manually from notebook

In [ ]:
client.call_auto(
    "starting_tests",
    sample_name="remote_test_001",
    t=2,
    setup=setup,
    indents=indents,
    remove_n=0,
    method_type="eh",
    use_import=0,
)

In [ ]:
client.call_auto("start_test_normal", t=2)

## Cell 11: Start test setup using imported indent array

In [ ]:
client.call_auto(
    "starting_tests",
    sample_name="remote_import_test_001",
    t=2,
    setup=setup,
    indents=None,
    remove_n=0,
    method_type="eh",
    use_import=1,
    import_dir=r"C:\Users\Public\Documents\Nanomechanics\Profiles\Shared AENI",
    import_file_name="indent_array.csv",
)

## Cell 12: ReviewData processing and export

In [ ]:
review_changes = {
    "poissons ratio": 0.34,
    "Drift": 1,
    "Min Depth for Results": 100,
    "Cite at Depth or Load": 0,
    "Stiffness for Contact": 1,
    "Depth to Cite Results (nm)": 500,
    "Load to cite Results (mN)": -1,
    "Metal : Pile up (0) or No Pile Up (1)": 1,
    "Test is Valid": 1,
}

review_changes

In [ ]:
result = client.call_auto(
    "process_review_file",
    sample_names=["remote_test_001"],
    open_dir=None,
    current_sample_file_name=None,
    export_dir=r"C:\Users\Public\Documents\Nanomechanics\Profiles\Shared AENI",
    export_file_name="remote_test_001_export",
    review_changes=review_changes,
    review_type="eh",
    export_format="excel",
    load_wait=30,
    recalc_wait=30,
    save_wait=5,
)

result

In [ ]:
export_path = client.download_last_export(local_dir="downloads")
export_path

## Cell 13: Open an existing sample in ReviewData and export

In [ ]:
result = client.call_auto(
    "process_review_file",
    sample_names=["existing_sample_name"],
    open_dir=r"C:\Users\Public\Documents\Nanomechanics\Profiles\Shared AENI",
    current_sample_file_name="existing_sample_name",
    export_dir=r"C:\Users\Public\Documents\Nanomechanics\Profiles\Shared AENI",
    export_file_name="existing_sample_export",
    review_changes=None,
    review_type="eh",
    export_format="csv",
)

result

In [ ]:
csv_path = client.download_last_export(local_dir="downloads")
csv_path

## Cell 14: Low-level approved calls

In [ ]:
ALLOWED_REMOTE_METHODS = [
    "process_review_file",
    "apply_review_parameters_ui",
    "click_review_button",
    "open_sample_from_directory",
    "save_review_file_as",
    "move",
    "move_in_increments",
    "starting_tests",
    "start_test_normal",
    "get_current_sample_and_test_ocr",
    "capture_video_panel",
    "capture_video_panel_remote",
    "change_method",
    "focus",
    "align_focus",
    "engage",
    "set_extension",
    "get_all_ui_state",
    "get_xyz_positions",
]

ALLOWED_REMOTE_METHODS

## Cell 15: Simple full workflow skeleton

In [ ]:
# 1. Check connection
print(client.ping())

# 2. Read initial position
print("Initial XYZ:", client.call_auto("get_xyz_positions"))

# 3. Change method
client.call_auto(
    "change_method",
    method_file_name="Advanced Dynamic E and H.NMT",
    add=1,
)

# 4. Setup tests
client.call_auto(
    "starting_tests",
    sample_name="demo_remote_run_001",
    setup=setup,
    indents=indents,
    method_type="eh",
    t=2,
)

# 5. Start test
client.call_auto("start_test_normal", t=2)

# 6. Read state
print("State:", client.call_auto("get_all_ui_state"))

# 7. Process ReviewData and export
client.call_auto(
    "process_review_file",
    sample_names=["demo_remote_run_001"],
    export_dir=r"C:\Users\Public\Documents\Nanomechanics\Profiles\Shared AENI",
    export_file_name="demo_remote_run_001_export",
    review_changes=review_changes,
    review_type="eh",
    export_format="excel",
)

# 8. Download export
downloaded = client.download_last_export(local_dir="downloads")
print("Downloaded:", downloaded)